# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
import gradio as gr
import openai
import base64
import tempfile
import subprocess
from dotenv import load_dotenv

load_dotenv()

# ==============================
# OpenAI Client
# ==============================

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ==============================
# Python Tool (Code Executor)
# ==============================

def run_python_code(code):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py") as tmp:
            tmp.write(code.encode())
            tmp_path = tmp.name

        result = subprocess.run(
            ["python", tmp_path],
            capture_output=True,
            text=True,
            timeout=10
        )

        return result.stdout if result.stdout else result.stderr

    except Exception as e:
        return str(e)


# ==============================
# Chat Function with Streaming
# ==============================

def chat_stream(user_input, history, model_choice):

    messages = [
        {
            "role": "system",
            "content": """
You are an expert Python teacher.
Only answer Python, programming, or technical questions.
Use headings and bullet points in Markdown.
If user asks to run code, call the python tool.
"""
        }
    ]

    for h in history:
        messages.append({"role": "user", "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})

    messages.append({"role": "user", "content": user_input})

    stream = client.chat.completions.create(
        model=model_choice,
        messages=messages,
        temperature=0.3,
        stream=True,
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "run_python_code",
                    "description": "Execute Python code",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "code": {"type": "string"}
                        },
                        "required": ["code"],
                    },
                },
            }
        ],
    )

    response = ""

    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response


# ==============================
# Audio Output Generator
# ==============================

def text_to_speech(text):
    speech_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text,
    ) as response:
        response.stream_to_file(speech_file.name)

    return speech_file.name


# ==============================
# Gradio UI
# ==============================

pink_theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="rose",
)

with gr.Blocks(theme=pink_theme) as demo:

    gr.Markdown("# 🌸 Python AI Chatbot")
    gr.Markdown("Expert Python Assistant with Streaming + Audio")

    model_dropdown = gr.Dropdown(
        choices=[
            "gpt-4o-mini",
            "gpt-4o",
            "gpt-4.1-mini",
        ],
        value="gpt-4o-mini",
        label="Select Model"
    )

    chatbot = gr.Chatbot()
    state = gr.State([])

    with gr.Row():
        txt = gr.Textbox(placeholder="Ask Python question...")
        audio_input = gr.Audio(type="filepath", label="Or Speak")

    send_btn = gr.Button("Send")

    audio_output = gr.Audio(label="Voice Response")

    # ==============================
    # Main Handler
    # ==============================

    def handle_input(text, audio, history, model):

        if audio:
            # Speech to Text
            transcript = client.audio.transcriptions.create(
                model="whisper-1",
                file=open(audio, "rb")
            )
            text = transcript.text

        history.append((text, ""))
        yield history, history, None

        bot_response = ""

        for chunk in chat_stream(text, history[:-1], model):
            bot_response = chunk
            history[-1] = (text, bot_response)
            yield history, history, None

        # Generate voice output
        audio_file = text_to_speech(bot_response)

        yield history, history, audio_file


    send_btn.click(
        handle_input,
        inputs=[txt, audio_input, state, model_dropdown],
        outputs=[chatbot, state, audio_output]
    )

demo.launch()



C:\Users\Hp\AppData\Local\Temp\ipykernel_7796\3600853197.py:136: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [1]:
import os
import gradio as gr
import openai
import tempfile
import subprocess
from dotenv import load_dotenv

load_dotenv()

# ==============================
# OpenAI Client
# ==============================

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ==============================
# Python Tool (Code Executor)
# ==============================

def run_python_code(code):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py") as tmp:
            tmp.write(code.encode())
            tmp_path = tmp.name

        result = subprocess.run(
            ["python", tmp_path],
            capture_output=True,
            text=True,
            timeout=10
        )

        return result.stdout if result.stdout else result.stderr

    except Exception as e:
        return str(e)


# ==============================
# Chat Streaming Function
# ==============================

def chat_stream(user_input, history, model_choice):

    messages = [
        {
            "role": "system",
            "content": """
You are an expert Python teacher.
Only answer Python, programming, or technical questions.
Use headings and bullet points in Markdown.
"""
        }
    ]

    # Add previous history safely
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    # Add new user message
    messages.append({"role": "user", "content": user_input})

    stream = client.chat.completions.create(
        model=model_choice,
        messages=messages,
        temperature=0.3,
        stream=True
    )

    response = ""

    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response


# ==============================
# Text To Speech
# ==============================

def text_to_speech(text):
    speech_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text,
    ) as response:
        response.stream_to_file(speech_file.name)

    return speech_file.name


# ==============================
# Main Input Handler
# ==============================

def handle_input(text, audio, history, model):

    history = history or []

    # 🎙️ Speech To Text
    if audio:
        try:
            with open(audio, "rb") as f:
                transcript = client.audio.transcriptions.create(
                    model="gpt-4o-mini-transcribe",
                    file=f
                )
            text = transcript.text
        except Exception as e:
            print("Transcription Error:", str(e))
            text = None

    if not text:
        yield history, history, None
        return

    # Create fresh history copy
    new_history = history + [(text, "")]
    yield new_history, new_history, None

    bot_response = ""

    # 🤖 Streaming response
    for chunk in chat_stream(text, history, model):
        bot_response = chunk
        new_history[-1] = (text, bot_response)
        yield new_history, new_history, None

    # 🔊 Text to Speech
    try:
        audio_file = text_to_speech(bot_response)
    except Exception as e:
        print("TTS Error:", str(e))
        audio_file = None

    yield new_history, new_history, audio_file


# ==============================
# Gradio UI
# ==============================

pink_theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="rose",
)

with gr.Blocks(theme=pink_theme) as demo:

    gr.Markdown("# 🌸 Python AI Chatbot")
    gr.Markdown("Expert Python Assistant with Streaming + Audio")

    model_dropdown = gr.Dropdown(
        choices=["gpt-4o-mini", "gpt-4o", "gpt-4.1-mini"],
        value="gpt-4o-mini",
        label="Select Model"
    )

    chatbot = gr.Chatbot()
    state = gr.State([])

    with gr.Row():
        txt = gr.Textbox(
            placeholder="Ask Python question...",
            scale=3
        )

        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Speak your question"
        )

    send_btn = gr.Button("Send")

    audio_output = gr.Audio(
        label="Voice Response",
        autoplay=True
    )

    send_btn.click(
        handle_input,
        inputs=[txt, audio_input, state, model_dropdown],
        outputs=[chatbot, state, audio_output]
    ).then(
        lambda: "",  # Clear textbox
        None,
        txt
    )

demo.launch()

C:\Users\Hp\AppData\Local\Temp\ipykernel_13548\4080495433.py:163: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [2]:
import os
import gradio as gr
from openai import OpenAI
import tempfile
import subprocess
import re
from dotenv import load_dotenv

load_dotenv()

# ============================
# OpenAI Client
# ============================

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# ============================
# Python Code Executor
# ============================

def run_python_code(code):
    try:
        with tempfile.NamedTemporaryFile(delete=False, suffix=".py", mode="w") as f:
            f.write(code)
            file_path = f.name

        result = subprocess.run(
            ["python", file_path],
            capture_output=True,
            text=True,
            timeout=10
        )

        output = result.stdout if result.stdout else result.stderr

        return output

    except Exception as e:
        return str(e)


# ============================
# Extract Python Code
# ============================

def extract_python_code(text):

    pattern = r"```python(.*?)```"
    matches = re.findall(pattern, text, re.DOTALL)

    if matches:
        return matches[0]

    return None


# ============================
# Chat Streaming
# ============================

def chat_stream(user_input, history, model):

    messages = [
        {
            "role": "system",
            "content": """
You are an expert Python teacher.

Rules:
- Answer programming questions clearly
- Use markdown formatting
- If code is required, wrap it inside ```python ```
"""
        }
    ]

    for user, bot in history:
        messages.append({"role": "user", "content": user})
        messages.append({"role": "assistant", "content": bot})

    messages.append({"role": "user", "content": user_input})

    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        stream=True
    )

    response = ""

    for chunk in stream:
        if chunk.choices[0].delta.content:
            response += chunk.choices[0].delta.content
            yield response


# ============================
# Text To Speech
# ============================

def text_to_speech(text):

    speech_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")

    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="alloy",
        input=text
    ) as response:

        response.stream_to_file(speech_file.name)

    return speech_file.name


# ============================
# Handle Input
# ============================

def handle_input(text, audio, history, model):

    history = history or []

    # 🎤 Speech to text
    if audio:
        with open(audio, "rb") as f:

            transcript = client.audio.transcriptions.create(
                model="gpt-4o-mini-transcribe",
                file=f
            )

        text = transcript.text

    if not text:
        yield history, history, None
        return

    new_history = history + [(text, "")]
    yield new_history, new_history, None

    bot_response = ""

    for chunk in chat_stream(text, history, model):

        bot_response = chunk
        new_history[-1] = (text, bot_response)

        yield new_history, new_history, None

    # ============================
    # Auto Python Code Execution
    # ============================

    code = extract_python_code(bot_response)

    if code:

        output = run_python_code(code)

        bot_response += f"\n\n### 🧪 Code Output\n```\n{output}\n```"

        new_history[-1] = (text, bot_response)

        yield new_history, new_history, None

    # ============================
    # Text To Speech
    # ============================

    try:

        audio_file = text_to_speech(bot_response)

    except:

        audio_file = None

    yield new_history, new_history, audio_file


# ============================
# Gradio UI
# ============================

pink_theme = gr.themes.Soft(
    primary_hue="pink",
    secondary_hue="rose"
)

with gr.Blocks(theme=pink_theme) as demo:

    gr.Markdown("#  Python AI Chatbot")
    gr.Markdown("Streaming + Voice + Python Code Execution")

    model_dropdown = gr.Dropdown(
        choices=["gpt-4o-mini", "gpt-4o", "gpt-4.1-mini"],
        value="gpt-4o-mini",
        label="Select Model"
    )

    chatbot = gr.Chatbot()

    state = gr.State([])

    with gr.Row():

        txt = gr.Textbox(
            placeholder="Ask Python question...",
            scale=3
        )

        audio_input = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Speak"
        )

    send_btn = gr.Button("Send")

    audio_output = gr.Audio(
        label="Voice Response",
        autoplay=True
    )

    send_btn.click(
        handle_input,
        inputs=[txt, audio_input, state, model_dropdown],
        outputs=[chatbot, state, audio_output]
    ).then(
        lambda: "",
        None,
        txt
    )

demo.launch()

C:\Users\Hp\AppData\Local\Temp\ipykernel_13548\3165869036.py:203: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
